In [1]:
print ("Hello world")

Hello world


In [2]:
import csv
import subprocess
from pathlib import Path

ROOT = Path("..").resolve()
FASTA = ROOT / "E-coli_data/ncbi_dataset/data/GCF_000005845.2/GCF_000005845.2_ASM584v2_genomic.fna"
OUT = ROOT / "results/stage0"

assert FASTA.exists(), FASTA


In [5]:
from Bio import SeqIO

def load_assembly(fasta_path):
    """Read a FASTA file into {contig_id: SeqRecord}."""
    contigs = SeqIO.to_dict(SeqIO.parse(fasta_path, "fasta"))
    if not contigs:
        raise ValueError(f"No sequences found in {fasta_path}")
    return contigs

contigs = load_assembly(FASTA)
print({name: len(record) for name, record in contigs.items()})


{'NC_000913.3': 4641652}


In [4]:
def assembly_stats(contigs):
    """Contig count, total length, N50/L50, GC% and ambiguous base count."""
    lengths = sorted((len(r) for r in contigs.values()), reverse=True)
    total = sum(lengths)
    running, n50, l50 = 0, 0, 0
    for i, length in enumerate(lengths, start=1):
        running += length
        if running >= total / 2:
            n50, l50 = length, i
            break
    counts = {b: sum(r.seq.upper().count(b) for r in contigs.values()) for b in "ACGT"}
    acgt = sum(counts.values())
    return {
        "n_contigs": len(lengths),
        "total_length": total,
        "longest_contig": lengths[0],
        "n50": n50,
        "l50": l50,
        "gc_percent": round(100 * (counts["G"] + counts["C"]) / acgt, 2),
        "ambiguous_bases": total - acgt,
    }


stats = assembly_stats(contigs)
stats


{'n_contigs': 1,
 'total_length': 4641652,
 'longest_contig': 4641652,
 'n50': 4641652,
 'l50': 1,
 'gc_percent': 50.79,
 'ambiguous_bases': 0}